# D232 - YARN: Run, Monitor, and Troubleshoot Applications

A practical guide for the single-node Hadoop 3.3.6 environment running inside Ubuntu on WSL.

> Run code cells from Jupyter inside WSL. Start HDFS and YARN before submitting applications.

Setup reference: [Hadoop on WSL](https://github.com/training-sh/hf-hyd/blob/main/setup/hadoop.md)

## 1. What YARN does

YARN manages cluster compute resources and schedules applications. HDFS stores data; YARN decides where and when computation runs.

| Component | Role |
|---|---|
| **ResourceManager** | Accepts applications and allocates cluster resources |
| **NodeManager** | Runs and monitors containers on one machine |
| **ApplicationMaster** | Coordinates one application and requests containers |
| **Container** | A resource allocation in which a task runs |
| **Client** | Submits, inspects, and controls an application |

On this single-node setup, the ResourceManager and NodeManager run as separate Java processes on the same WSL instance.

## 2. Start the Hadoop services in WSL

Open an Ubuntu/WSL terminal. Start SSH first because Hadoop's helper scripts connect to the configured worker host.

In [ ]:
%%bash
sudo service ssh start
start-dfs.sh
start-yarn.sh
jps

Expected processes:

- HDFS: `NameNode`, `DataNode`, `SecondaryNameNode`
- YARN: `ResourceManager`, `NodeManager`
- Later: `JobHistoryServer`

`jps` proves that Java processes exist. The commands below also test whether YARN can report its nodes and queue.

In [ ]:
%%bash
yarn version
echo '=== Running NodeManagers ==='
yarn node -list
echo '=== Default queue ==='
yarn queue -status default

## 3. Web interfaces

Open these URLs in the Windows browser while Hadoop is running in WSL:

| Interface | URL | What to inspect |
|---|---|---|
| ResourceManager | [http://localhost:8088](http://localhost:8088) | applications, states, queues, nodes, memory, vCores |
| ResourceManager applications | [http://localhost:8088/cluster/apps](http://localhost:8088/cluster/apps) | running and completed applications |
| ResourceManager nodes | [http://localhost:8088/cluster/nodes](http://localhost:8088/cluster/nodes) | NodeManager health and resources |
| NodeManager | [http://localhost:8042](http://localhost:8042) | containers and local logs on this node |
| MapReduce JobHistory | [http://localhost:19888](http://localhost:19888) | completed MapReduce jobs, tasks, attempts, counters, logs |
| NameNode | [http://localhost:9870](http://localhost:9870) | HDFS files and cluster storage |

WSL normally forwards listening ports to Windows `localhost`. If a page does not open, confirm the daemon with `jps` and check the port with `ss -ltnp`.

## 4. Prepare HDFS input

MapReduce input and output paths in this notebook are HDFS paths. An output directory must not already exist when a job starts.

In [ ]:
%%bash
LAB="/user/$USER/d232"
hdfs dfs -mkdir -p "$LAB/input"
printf '%s\n' \
  'yarn manages cluster resources' \
  'hdfs stores data and yarn schedules work' \
  'containers run application tasks' \
  | hdfs dfs -put -f - "$LAB/input/words.txt"
hdfs dfs -ls -R "$LAB"
hdfs dfs -cat "$LAB/input/words.txt"

## 5. Submit a MapReduce application

The Hadoop examples JAR is included with Hadoop. WordCount gives us a small application to inspect through YARN.

In [ ]:
%%bash
LAB="/user/$USER/d232"
EXAMPLES_JAR=$(find "$HADOOP_HOME/share/hadoop/mapreduce" -maxdepth 1 -name 'hadoop-mapreduce-examples-*.jar' | head -1)
test -n "$EXAMPLES_JAR" || { echo 'Examples JAR not found'; exit 1; }
hdfs dfs -rm -r -f "$LAB/output-wordcount"
hadoop jar "$EXAMPLES_JAR" wordcount "$LAB/input" "$LAB/output-wordcount"
hdfs dfs -cat "$LAB/output-wordcount/part-r-*"

### Submit a job that runs long enough to monitor

The `pi` example is useful for observing an active application. Increase the map count or samples if it finishes too quickly. Keep a second WSL terminal ready for monitoring commands.

In [ ]:
%%bash
EXAMPLES_JAR=$(find "$HADOOP_HOME/share/hadoop/mapreduce" -maxdepth 1 -name 'hadoop-mapreduce-examples-*.jar' | head -1)
hadoop jar "$EXAMPLES_JAR" pi 20 100000

`hadoop jar` and `yarn jar` can both launch a JAR. Hadoop MapReduce jobs commonly use `hadoop jar`. The configured `mapreduce.framework.name=yarn` makes the job run through YARN.

## 6. List applications by state

YARN does not use `PENDING` as an application state. An application waiting in the scheduler is normally **ACCEPTED**. Once containers begin executing, it becomes **RUNNING**.

In [ ]:
%%bash
echo '=== Submitted and waiting for resources ==='
yarn application -list -appStates SUBMITTED,ACCEPTED
echo '=== Running ==='
yarn application -list -appStates RUNNING
echo '=== Finished, failed, or killed ==='
yarn application -list -appStates FINISHED,FAILED,KILLED
echo '=== All applications ==='
yarn application -list -appStates ALL

Application lifecycle:

`NEW → SUBMITTED → ACCEPTED → RUNNING → FINISHED | FAILED | KILLED`

Capture the application ID printed at submission, or copy it from `yarn application -list`. It looks like `application_..._0001`.

## 7. Inspect an application, attempt, and containers

Set `APP_ID` before running these cells. Do not include angle brackets.

In [ ]:
%%bash
APP_ID='application_XXXXXXXXXXXXX_0001'
yarn application -status "$APP_ID"
yarn applicationattempt -list "$APP_ID"

Copy the `ApplicationAttempt-Id` from the previous output, then list its containers.

In [ ]:
%%bash
ATTEMPT_ID='appattempt_XXXXXXXXXXXXX_0001_000001'
yarn applicationattempt -status "$ATTEMPT_ID"
yarn container -list "$ATTEMPT_ID"
# After copying a container ID:
# yarn container -status container_XXXXXXXXXXXXX_0001_01_000001

## 8. NodeManager commands

A NodeManager reports node health and available resources to the ResourceManager. It launches containers and keeps their local logs until cleanup.

In [ ]:
%%bash
echo '=== Running nodes ==='
yarn node -list
echo '=== Nodes in every state ==='
yarn node -list -all
echo '=== Unhealthy or lost nodes ==='
yarn node -list -states UNHEALTHY,LOST
echo '=== NodeManager daemon log files ==='
ls -1 "${HADOOP_LOG_DIR:-$HADOOP_HOME/logs}"/*nodemanager* 2>/dev/null || true

To inspect one node, copy the `Node-Id` returned by `yarn node -list`:

```bash
yarn node -status HOSTNAME:PORT
```

Start or stop only the local NodeManager when diagnosing it:

```bash
yarn --daemon stop nodemanager
yarn --daemon start nodemanager
```

For the complete single-node service, prefer `stop-yarn.sh` and `start-yarn.sh`.

## 9. Kill an application

Kill only an application you own and no longer need. Its final state becomes `KILLED`; partial output may remain in HDFS.

In [ ]:
%%bash
APP_ID='application_XXXXXXXXXXXXX_0001'
yarn application -status "$APP_ID"
# Uncomment after checking the ID:
# yarn application -kill "$APP_ID"
# yarn application -status "$APP_ID"

## 10. Application logs from the command line

`yarn logs` is the preferred interface. It hides whether logs are still local or have been aggregated into remote storage. Run it after the application finishes when log aggregation is enabled.

In [ ]:
%%bash
APP_ID='application_XXXXXXXXXXXXX_0001'
echo '=== All aggregated container logs ==='
yarn logs -applicationId "$APP_ID"
echo '=== Only stderr, when supported by this Hadoop version ==='
yarn logs -applicationId "$APP_ID" -log_files stderr

Useful variants:

```bash
yarn logs -applicationId APP_ID -appOwner USERNAME
yarn logs -applicationId APP_ID -containerId CONTAINER_ID
yarn logs -applicationId APP_ID -log_files stdout,stderr
yarn logs -applicationId APP_ID -log_files ALL
```

Common files are `stdout`, `stderr`, and `syslog`. An application may produce several containers, so the full command can return several copies of each log type.

## 11. View logs in the web interfaces

### While an application is running
1. Open [ResourceManager applications](http://localhost:8088/cluster/apps).
2. Select the application ID.
3. Open the application attempt.
4. Select a container.
5. Follow the log link to the NodeManager page.

### After a MapReduce job finishes
1. Open [MapReduce JobHistory](http://localhost:19888).
2. Select the completed job.
3. Open a map or reduce task, then its attempt.
4. Select `logs` to view `stdout`, `stderr`, and `syslog`.

A log link can fail after container cleanup if aggregation is disabled or the JobHistory/log server is not configured.

## 12. Log aggregation and HDFS

With log aggregation enabled, each NodeManager uploads completed container logs to remote storage. In this setup the remote filesystem is normally HDFS.

Key `yarn-site.xml` setting:

```xml
<property>
  <name>yarn.log-aggregation-enable</name>
  <value>true</value>
</property>
```

Restart YARN after changing its configuration. The common default remote root is `/tmp/logs`, but always confirm the actual cluster configuration.

In [ ]:
%%bash
echo '=== Relevant configured properties ==='
grep -n -A2 -B1 -E 'yarn.log-aggregation-enable|yarn.nodemanager.remote-app-log-dir|yarn.nodemanager.remote-app-log-dir-suffix|yarn.nodemanager.log-dirs' \
  "$HADOOP_HOME/etc/hadoop/yarn-site.xml" || true
echo '=== Common HDFS aggregated-log root ==='
hdfs dfs -ls /tmp/logs 2>/dev/null || echo '/tmp/logs does not exist or is not accessible'

### Find and retrieve an aggregated log from HDFS

Aggregated files are container log bundles, not always plain text. Use `yarn logs` to decode them. Direct HDFS access is useful for confirming location, permissions, and file presence.

In [ ]:
%%bash
APP_ID='application_XXXXXXXXXXXXX_0001'
echo '=== Locate this application under the remote log root ==='
hdfs dfs -find /tmp/logs -name "$APP_ID" -print 2>/dev/null || true
echo '=== Inspect matching files and directories ==='
hdfs dfs -ls -R /tmp/logs 2>/dev/null | grep "$APP_ID" || true
echo '=== Decode the application logs through YARN ==='
yarn logs -applicationId "$APP_ID"
# To copy an identified HDFS log bundle for investigation:
# hdfs dfs -get HDFS_LOG_FILE /tmp/

If `yarn logs` says logs are unavailable, check:

- the application ID and owner are correct
- the application has finished
- `yarn.log-aggregation-enable` is `true`
- the NodeManager could write to the remote log directory
- HDFS is running and the remote log path exists
- permissions allow the current user to read the logs
- NodeManager logs contain no aggregation errors

## 13. Role of the MapReduce JobHistoryServer

The ResourceManager manages cluster resources and current application state. It is not the detailed long-term history service for completed MapReduce jobs.

The **JobHistoryServer** provides:

- completed MapReduce jobs
- map and reduce task attempts
- counters and diagnostics
- links to aggregated container logs

It reads MapReduce history files written to configured history locations. It does not run tasks and it is not required for a job to execute.

In [ ]:
%%bash
echo '=== Start JobHistoryServer ==='
mapred --daemon start historyserver
jps | grep -E 'JobHistoryServer|Jps'
echo 'Open http://localhost:19888'
echo '=== MapReduce history configuration, if explicitly set ==='
grep -n -A2 -B1 -E 'mapreduce.jobhistory.(address|webapp.address|done-dir|intermediate-done-dir)' \
  "$HADOOP_HOME/etc/hadoop/mapred-site.xml" || true

Stop it when needed:

```bash
mapred --daemon stop historyserver
```

If port `19888` does not open, inspect the JobHistoryServer daemon log in `${HADOOP_LOG_DIR:-$HADOOP_HOME/logs}` and confirm the history directories are writable.

## 14. Fast troubleshooting

| Symptom | First checks |
|---|---|
| Application remains `ACCEPTED` | `yarn node -list`, queue capacity, requested memory/vCores |
| No NodeManagers | `jps`, NodeManager daemon log, `yarn-site.xml`, hostname resolution |
| Application fails immediately | `yarn application -status APP_ID`, `yarn logs -applicationId APP_ID` |
| Output path already exists | remove or choose a new HDFS output directory |
| ResourceManager UI unavailable | `jps`, `ss -ltnp | grep 8088`, ResourceManager daemon log |
| NodeManager log link fails | try `yarn logs`; check aggregation and port `8042` |
| Completed job missing from history | start JobHistoryServer; inspect history configuration and logs |
| HDFS logs missing | verify aggregation, remote directory, permissions, and NodeManager errors |

In [ ]:
%%bash
echo '=== Java daemons ==='
jps
echo '=== Service ports ==='
ss -ltn 2>/dev/null | grep -E ':(8088|8042|19888|9870)\b' || true
echo '=== Nodes and running applications ==='
yarn node -list
yarn application -list -appStates RUNNING,ACCEPTED
echo '=== Recent YARN daemon errors ==='
grep -h -i -E 'error|exception|fatal' "${HADOOP_LOG_DIR:-$HADOOP_HOME/logs}"/*resourcemanager* "${HADOOP_LOG_DIR:-$HADOOP_HOME/logs}"/*nodemanager* 2>/dev/null | tail -30 || true

## Command reference

| Task | Command |
|---|---|
| Start/stop YARN | `start-yarn.sh`, `stop-yarn.sh` |
| List applications | `yarn application -list -appStates ALL` |
| Waiting applications | `yarn application -list -appStates SUBMITTED,ACCEPTED` |
| Running applications | `yarn application -list -appStates RUNNING` |
| Application details | `yarn application -status APP_ID` |
| Kill application | `yarn application -kill APP_ID` |
| List attempts | `yarn applicationattempt -list APP_ID` |
| List containers | `yarn container -list ATTEMPT_ID` |
| List nodes | `yarn node -list -all` |
| Node details | `yarn node -status NODE_ID` |
| Queue details | `yarn queue -status default` |
| Application logs | `yarn logs -applicationId APP_ID` |
| Start/stop history | `mapred --daemon start historyserver`, `mapred --daemon stop historyserver` |
| ResourceManager UI | [http://localhost:8088](http://localhost:8088) |
| NodeManager UI | [http://localhost:8042](http://localhost:8042) |
| JobHistory UI | [http://localhost:19888](http://localhost:19888) |

## References

- [Course Hadoop on WSL setup](https://github.com/training-sh/hf-hyd/blob/main/setup/hadoop.md)
- [Apache Hadoop: Single Node Setup](https://hadoop.apache.org/docs/stable/hadoop-project-dist/hadoop-common/SingleCluster.html)
- [Apache Hadoop: YARN Commands](https://hadoop.apache.org/docs/stable/hadoop-yarn/hadoop-yarn-site/YarnCommands.html)
- [Apache Hadoop: MapReduce Commands](https://hadoop.apache.org/docs/stable/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapredCommands.html)
- [Apache Hadoop: MapReduce Tutorial](https://hadoop.apache.org/docs/stable/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html)